# Manhattan Reasoning Gym — run an agent in a sandbox

The RL-gym side: an **agent** iterates on a design *inside a locked-down
container* (no network, no credentials), and when it likes a candidate it
**promotes** it to silicon. You declare a **`Sandbox`** on the host and run it;
the promote is brokered to silicon internally. This notebook runs that whole
loop with a **no-op silicon backend** (no key, no real board).

Two vantage points:
- **inside the sandbox (agent):** `mrg.build` + `mrg.sandbox`
- **outside on the host (operator):** `mrg.bench` (`mrg.Sandbox`)

**Prerequisites:** Python + Docker + the `ghcr.io/barnard-pl-labs/mrg-sandbox:latest` image (see notebook 1).

## 1. Install the SDK

In [ ]:
# Locate the repo root (works from anywhere inside the checkout).
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
print("repo root:", ROOT)

In [ ]:
# Install the mrg SDK (one package). Install the SDK from this checkout.
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)],
               check=True)
print("installed manhattan-reasoning-gym")

## 2. The agent

This is the code that runs *inside* the sandbox. It uses `mrg.build` for local
feedback and, only after it likes its candidate, `mrg.sandbox.promote` to request
silicon. It has no key and no network. **The agent decides what to promote** —
the host imposes no gate unless you pass one.

In [ ]:
AGENT = ROOT / "examples" / "agent.py"
print(AGENT.read_text())

## 3. The operator declares a sandbox and runs it — `mrg.Sandbox`

On the host you declare a **`Sandbox`** with your files and call `.run()`. It
launches the agent in a **locked** container (`--network none`, no key) and
brokers the agent's promote to silicon internally — there's no broker to stand
up. With no key set, `silicon="auto"` is a no-op backend. The container and host
talk only through a shared workspace the `Sandbox` manages for you.

In [ ]:
import manhattan_reasoning_gym as mrg

# Declare the sandbox with your files, then run the agent. Locked profile is the
# default (--network none, no key). silicon="auto" with no key => no-op backend.
sandbox = mrg.Sandbox(files=[ROOT / "examples/design.py", AGENT])
result = sandbox.run("agent.py", timeout=900)

print(result.stdout)
for promo in result.promotions:
    print("promote:", promo["accepted"], promo.get("silicon", promo.get("reason")))

### What happened

- The agent ran **offline** in a locked container (no network, no key) and used
  `mrg.build` for synth/pnr feedback.
- It `promote`d a candidate; the `Sandbox` brokered it to (no-op) silicon and
  returned the outcome in `result.promotions`. No host-side gating was applied.
- The trust boundary is structural: the untrusted container can't reach the
  cloud or hold a key — only the host `Sandbox` can.

**To use real boards**, ask for the cloud backend (the key stays on the host,
never in the container):

```python
sandbox = mrg.Sandbox(files=[...], silicon="cloud")   # uses MRG_API_KEY
```

**To gate promotes on the host** (optional — off by default), pass a `guard`:

```python
def guard(design_bytes, report):
    return None if report.get("timing_met") else "timing_not_met"

sandbox = mrg.Sandbox(files=[...], guard=guard)
```